In [3]:
import pandas as pd
import requests
from datetime import datetime,timedelta
from dateutil.relativedelta import relativedelta

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11676\3731028418.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [4]:
current_date = datetime.now()
formatted_date = current_date.strftime("%Y%m%d")
date=timedelta(1)+datetime.now()
print(formatted_date,date)

20240702 2024-07-03 17:12:12.198925


In [5]:
def quater(date):
    if date.month>=1 and date.month<=3:
      return {'pyt_fetch':str((date-relativedelta(years=1)).year)+'1001T000000','db_fetch_start':str((date-relativedelta(years=1)).year)+'0701','db_fetch_end':str((date-relativedelta(years=1)).year)+'0930'}
    elif date.month>=4 and date.month<=6:
      return {'pyt_fetch':str(date.year)+'0101T000000','db_fetch_start':str((date-relativedelta(years=1)).year)+'1001','db_fetch_end':str((date-relativedelta(years=1)).year)+'1231'}
    elif date.month>=7 and date.month<=9:
      return {'pyt_fetch':str(date.year)+'0401T000000','db_fetch_start':str(date.year)+'0101','db_fetch_end':str(date.year)+'0331'}
    else:
      return {'pyt_fetch':str(date.year)+'0701T000000','db_fetch_start':str(date.year)+'0401','db_fetch_end':str(date.year)+'0630'}

quat=quater(current_date)   


In [6]:
# Your credentials
username = 'MondelezPortland'
password = 'BakerySLX360'

# URL and parameters
url = "https://saas75.shoplogix.com/web/api/export/summary"
params = {
    'start': quat['db_fetch_start'],
    'end': quat['db_fetch_end'],
    'metrics': 'comments',
    'machines': '862C993B-C83D-1AE6-4191-A278302D7B59,C7CC9488-34B8-4654-35D6-6D4B073DB900,398F6011-E432-5D96-AE00-BC601454EFC9,58E4DF72-A0BA-9FCC-1B3E-4E1B362C29FD,6922AF33-1EAF-6C10-F8BB-5E4350895477,E734E668-C538-05D8-48E4-A27AAD11969A,1C0A3C71-CAAA-894B-CD18-71FC05CB55B2,8A40D3AD-50DA-6409-051B-5E8BF0C0AFEA,56BD02A6-D1D0-6CA7-58E9-0E595CDDE9CE,93B6A077-27F4-0EBE-BD99-0E595CE36496,8D4D54E7-CE42-589C-52B4-4F0910520C83',
    'groupBy': 'Machine,shiftInstance'
}

# Making the GET request with Basic Auth
try:
    response = requests.get(url, params=params, auth=(username, password))
    response.raise_for_status()  # Raises stored HTTPError, if one occurred
    data=response.json()
    # Output the JSON response
    print(response.json())
except requests.exceptions.HTTPError as errh:
    print(f"HTTP Error: {errh}")
except requests.exceptions.ConnectionError as errc:
    print(f"Error Connecting: {errc}")
except requests.exceptions.Timeout as errt:
    print(f"Timeout Error: {errt}")
except requests.exceptions.RequestException as err:
    print(f"OOps: Something Else: {err}")


{'result': {'query': 'https://saas75.shoplogix.com:443/web/api/export/summary?start=20240101&end=20240331&metrics=comments&machines=862C993B-C83D-1AE6-4191-A278302D7B59%2CC7CC9488-34B8-4654-35D6-6D4B073DB900%2C398F6011-E432-5D96-AE00-BC601454EFC9%2C58E4DF72-A0BA-9FCC-1B3E-4E1B362C29FD%2C6922AF33-1EAF-6C10-F8BB-5E4350895477%2CE734E668-C538-05D8-48E4-A27AAD11969A%2C1C0A3C71-CAAA-894B-CD18-71FC05CB55B2%2C8A40D3AD-50DA-6409-051B-5E8BF0C0AFEA%2C56BD02A6-D1D0-6CA7-58E9-0E595CDDE9CE%2C93B6A077-27F4-0EBE-BD99-0E595CE36496%2C8D4D54E7-CE42-589C-52B4-4F0910520C83&groupBy=Machine%2CshiftInstance', 'machines': [{'machineId': '862C993B-C83D-1AE6-4191-A278302D7B59', 'machineName': 'L18-Dough Machine 1 - BN', 'erpCode': '', 'shiftInstances': [{'shiftName': 'Shift 3', 'shiftInstance': '862C993B-C83D-1AE6-4191-A278302D7B59 Shift 3 2023-12-31 23:15:00', 'start': '20240101T000000.000', 'end': '20240101T071500.000', 'metrics': {'comments': [{'machineId': '862C993B-C83D-1AE6-4191-A278302D7B59', 'start': '20

In [7]:

flattened_data = []

def flatten_data(machine):
    machine_id = machine['machineId']
    machine_name = machine['machineName']
    erp_code = machine['erpCode']
    
    for instance in machine['shiftInstances']:
        shift_name = instance['shiftName']
        shift_instance = instance['shiftInstance']
        start = instance['start']
        end = instance['end']
        
        for comment in instance['metrics']['comments']:
            # Check if 'duration' is present and not empty
            if 'duration' in comment and comment['duration']:
                comment_details = {
                    'machineId': machine_id,
                    'machineName': machine_name,
                    'erpCode': erp_code,
                    'shiftName': shift_name,
                    'shiftInstance': shift_instance,
                    'start': start,
                    'end': end,
                    'commentStart': comment.get('start'),
                    'commentEnd': comment.get('end'),
                    'comment': comment.get('comment'),
                    'duration': comment['duration'],  
                    'type': comment.get('type', 'Unknown'),
                    'reason': comment.get('reason', 'Not specified')
                }
                flattened_data.append(comment_details)
            else:
                continue  # Skip this comment if duration is missing or empty

# Iterate over each machine in the JSON data
for machine in data['result']['machines']:
    flatten_data(machine)

# Create a DataFrame from the collected data
df1 = pd.DataFrame(flattened_data)
df1

,machineId,machineName,erpCode,shiftName,shiftInstance,start,end,commentStart,commentEnd,comment,duration,type,reason
0,862C993B-C83D-1AE6-4191-A278302D7B59,L18-Dough Machine 1 - BN,,Shift 3,862C993B-C83D-1AE6-4191-A278302D7B59 Shift 3 2...,20240101T000000.000,20240101T071500.000,20240101T061500.000,20240101T001500.000,No Demand,7.250000,Idle,Lack of Market Demand
1,862C993B-C83D-1AE6-4191-A278302D7B59,L18-Dough Machine 1 - BN,,Shift 1,862C993B-C83D-1AE6-4191-A278302D7B59 Shift 1 2...,20240101T071500.000,20240101T151500.000,20240101T071500.000,20240101T091500.000,No Demand,8.000000,Idle,Lack of Market Demand
2,862C993B-C83D-1AE6-4191-A278302D7B59,L18-Dough Machine 1 - BN,,Shift 2,862C993B-C83D-1AE6-4191-A278302D7B59 Shift 2 2...,20240101T151500.000,20240101T231500.000,20240101T151500.000,20240101T161500.000,No Demand,8.000000,Idle,Lack of Market Demand
3,862C993B-C83D-1AE6-4191-A278302D7B59,L18-Dough Machine 1 - BN,,Shift 3,862C993B-C83D-1AE6-4191-A278302D7B59 Shift 3 2...,20240101T231500.000,20240102T071500.000,20240102T061500.000,20240102T001500.000,No Demand,7.884884,Idle,Lack of Market Demand
4,862C993B-C83D-1AE6-4191-A278302D7B59,L18-Dough Machine 1 - BN,,Shift 1,862C993B-C83D-1AE6-4191-A278302D7B59 Shift 1 2...,20240102T071500.000,20240102T151500.000,20240102T091500.000,20240102T121500.000,No Demand,3.636973,Idle,Lack of Market Demand
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3899,8D4D54E7-CE42-589C-52B4-4F0910520C83,L05-Case Sealer,,Shift 1,8D4D54E7-CE42-589C-52B4-4F0910520C83 Shift 1 2...,20240328T071500.000,20240328T151500.000,20240328T111500.000,20240328T121500.000,90 overhead conveyor was spilling cookies onto...,0.005556,Micro Stoppage,BD - Electrical Case Conveying
3900,8D4D54E7-CE42-589C-52B4-4F0910520C83,L05-Case Sealer,,Shift 1,8D4D54E7-CE42-589C-52B4-4F0910520C83 Shift 1 2...,20240328T071500.000,20240328T151500.000,20240328T121500.000,20240328T131500.000,Open fin seal. ACTION: Made manual adjustment....,0.001389,Starved,BD - Ln5-Wrapper 2
3901,8D4D54E7-CE42-589C-52B4-4F0910520C83,L05-Case Sealer,,Shift 1,8D4D54E7-CE42-589C-52B4-4F0910520C83 Shift 1 2...,20240328T071500.000,20240328T151500.000,20240328T141500.000,20240328T151500.000,No Demand,0.093360,Starved,Lack of Market Demand
3902,8D4D54E7-CE42-589C-52B4-4F0910520C83,L05-Case Sealer,,Shift 1,8D4D54E7-CE42-589C-52B4-4F0910520C83 Shift 1 2...,20240328T071500.000,20240328T151500.000,20240328T141500.000,20240328T151500.000,No Demand,0.544462,Faulted,Lack of Market Demand


In [18]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3904 entries, 0 to 3903
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   machineId      3904 non-null   object 
 1   machineName    3904 non-null   object 
 2   erpCode        3904 non-null   object 
 3   shiftName      3904 non-null   object 
 4   shiftInstance  3904 non-null   object 
 5   start          3904 non-null   object 
 6   end            3904 non-null   object 
 7   commentStart   3904 non-null   object 
 8   commentEnd     3904 non-null   object 
 9   comment        3904 non-null   object 
 10  duration       3904 non-null   float64
 11  type           3904 non-null   object 
 12  reason         3904 non-null   object 
dtypes: float64(1), object(12)
memory usage: 396.6+ KB


In [8]:
df1['duration'] = pd.to_numeric(df1['duration'], errors='coerce')
df1.info()
df1=df1.drop(['erpCode','shiftInstance','commentStart','commentEnd'],axis=1)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3904 entries, 0 to 3903
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   machineId      3904 non-null   object 
 1   machineName    3904 non-null   object 
 2   erpCode        3904 non-null   object 
 3   shiftName      3904 non-null   object 
 4   shiftInstance  3904 non-null   object 
 5   start          3904 non-null   object 
 6   end            3904 non-null   object 
 7   commentStart   3904 non-null   object 
 8   commentEnd     3904 non-null   object 
 9   comment        3904 non-null   object 
 10  duration       3904 non-null   float64
 11  type           3904 non-null   object 
 12  reason         3904 non-null   object 
dtypes: float64(1), object(12)
memory usage: 396.6+ KB


In [24]:
import pymysql
connection=pymysql.connect(
    host="s13.hosterpk.com",
    user="digita87_muddassir",
    password="Developers000$$$",
    database="digita87_mondeleez"
)
data_tuples=list(df1.itertuples(index=False,name=None))
insert_query="""INSERT INTO comments (machineId,machinename,shiftname,start,end,comment,duration,type,reason)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)"""
with connection.cursor() as cursor:
    cursor.executemany(insert_query,data_tuples)
connection.commit() 

connection.close()
print("Data inserted successfully!")

Data inserted successfully!


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11676\2695688003.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df2=pd.read_sql_query("SELECT * FROM comments",connection)


,machineId,machineName,shiftName,start,end,comment,duration,type,reason
0,862C993B-C83D-1AE6-4191-A278302D7B59,L18-Dough Machine 1 - BN,Shift 3,20240101T000000.000,20240101T071500.000,No Demand,7.250000,Idle,Lack of Market Demand
1,862C993B-C83D-1AE6-4191-A278302D7B59,L18-Dough Machine 1 - BN,Shift 1,20240101T071500.000,20240101T151500.000,No Demand,8.000000,Idle,Lack of Market Demand
2,862C993B-C83D-1AE6-4191-A278302D7B59,L18-Dough Machine 1 - BN,Shift 2,20240101T151500.000,20240101T231500.000,No Demand,8.000000,Idle,Lack of Market Demand
3,862C993B-C83D-1AE6-4191-A278302D7B59,L18-Dough Machine 1 - BN,Shift 3,20240101T231500.000,20240102T071500.000,No Demand,7.884884,Idle,Lack of Market Demand
4,862C993B-C83D-1AE6-4191-A278302D7B59,L18-Dough Machine 1 - BN,Shift 1,20240102T071500.000,20240102T151500.000,No Demand,3.636973,Idle,Lack of Market Demand
...,...,...,...,...,...,...,...,...,...
7803,8D4D54E7-CE42-589C-52B4-4F0910520C83,L05-Case Sealer,Shift 1,20240328T071500.000,20240328T151500.000,90 overhead conveyor was spilling cookies onto...,0.005556,Micro Stoppage,BD - Electrical Case Conveying
7804,8D4D54E7-CE42-589C-52B4-4F0910520C83,L05-Case Sealer,Shift 1,20240328T071500.000,20240328T151500.000,Open fin seal. ACTION: Made manual adjustment....,0.001389,Starved,BD - Ln5-Wrapper 2
7805,8D4D54E7-CE42-589C-52B4-4F0910520C83,L05-Case Sealer,Shift 1,20240328T071500.000,20240328T151500.000,No Demand,0.093360,Starved,Lack of Market Demand
7806,8D4D54E7-CE42-589C-52B4-4F0910520C83,L05-Case Sealer,Shift 1,20240328T071500.000,20240328T151500.000,No Demand,0.544462,Faulted,Lack of Market Demand
